In [1]:
"""
Knowledge Distillation — Script 6: Cross-Architecture Distillation (Best Config Only)
======================================================================================
Teacher : ResNet-50 (modified for CIFAR-10, pre-trained baseline)
Student : MobileNetV2 (modified for CIFAR-10)
Dataset : CIFAR-10
Method  : CKA-Guided Projection Distillation — proj_dim=64, beta=1.0, tau=4.0, epochs=10

A shared projection head maps BOTH teacher and student features into a common
latent space R^d, then applies MSE there:
  L = CE + beta * ||g_T(F_T) - g_S(F_S)||_F^2
where g_T, g_S are lightweight MLP projectors (in_dim→256→proj_dim, L2-normalised).
CKA is logged per epoch to monitor representational alignment (not used in loss).

Output : __6__KD_cross_architecture_proj_best.json
"""

import os, json, time, copy, tempfile, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

warnings.filterwarnings("ignore")

# ── Device ─────────────────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[init] PyTorch {torch.__version__}", flush=True)
print(f"[init] Device  : {DEVICE}", flush=True)
if DEVICE.type == "cuda":
    print(f"[init] GPU     : {torch.cuda.get_device_name(0)}", flush=True)

# ── Config ─────────────────────────────────────────────────────────────────────
TEACHER_MODEL_PATH    = "../../__2__baseline_resnet50_cifar10.pth"
BASELINE_METRICS_PATH = "../../__2__baseline_metrics.json"
OUTPUT_JSON           = "__6__KD_cross_architecture_proj_best.json"

BATCH_SIZE  = 128 if DEVICE.type == "cuda" else 64
NUM_WORKERS = 0
PIN_MEMORY  = False
NUM_CLASSES = 10
USE_AMP     = (DEVICE.type == "cuda")

# Best config (from prior sweep)
METHOD    = "projection"
TAU       = 4.0
ALPHA     = 0.1
BETA      = 1.0
GAMMA     = 0.5
PROJ_DIM  = 64
PROJ_TEMP = 0.07
LR        = 0.01
KD_EPOCHS = 10

# Internal embedding dims (architecture-fixed)
T_DIM = 2048   # ResNet-50 avgpool output
S_DIM = 1280   # MobileNetV2 last features output

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2023, 0.1994, 0.2010)

print(f"[init] AMP     : {USE_AMP}", flush=True)
print(f"[init] Config  : {METHOD} | proj_dim={PROJ_DIM} | beta={BETA} | "
      f"tau={TAU} | epochs={KD_EPOCHS}", flush=True)


# ── Model builders ─────────────────────────────────────────────────────────────
def build_resnet50_cifar(num_classes: int = 10) -> nn.Module:
    model = models.resnet50(weights=None)
    model.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc      = nn.Linear(model.fc.in_features, num_classes)
    return model

def build_mobilenetv2_cifar(num_classes: int = 10) -> nn.Module:
    model = models.mobilenet_v2(weights=None)
    model.features[0][0] = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1, bias=False)
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)
    return model

def load_teacher(path: str) -> nn.Module:
    print(f"[load] Loading teacher from {path} ...", flush=True)
    model = build_resnet50_cifar(NUM_CLASSES)
    model.load_state_dict(torch.load(path, map_location="cpu"))
    model.eval().to(DEVICE)
    print("[load] Teacher OK", flush=True)
    return model


# ── Embedding extractors ───────────────────────────────────────────────────────
class EmbeddingExtractor(nn.Module):
    """Hooks avgpool of ResNet-50 → 2048-d embedding."""
    def __init__(self, model: nn.Module):
        super().__init__()
        self.model  = model
        self._embed = None
        self._hook  = model.avgpool.register_forward_hook(
            lambda m, i, o: setattr(self, "_embed", o.view(o.size(0), -1))
        )

    def forward(self, x):
        logits = self.model(x)
        return logits, self._embed

    def remove_hook(self): self._hook.remove()


class MobileNetEmbeddingExtractor(nn.Module):
    """Hooks last features block of MobileNetV2 → 1280-d embedding."""
    def __init__(self, model: nn.Module):
        super().__init__()
        self.model  = model
        self._embed = None
        self._hook  = model.features.register_forward_hook(
            lambda m, i, o: setattr(
                self, "_embed",
                F.adaptive_avg_pool2d(o, 1).view(o.size(0), -1)
            )
        )

    def forward(self, x):
        logits = self.model(x)
        return logits, self._embed

    def remove_hook(self): self._hook.remove()


# ── Projection head ────────────────────────────────────────────────────────────
class ProjectionHead(nn.Module):
    """MLP: in_dim → 256 → proj_dim (L2-normalised output)."""
    def __init__(self, in_dim: int, proj_dim: int = 64, hidden: int = 256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.BatchNorm1d(hidden),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, proj_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return F.normalize(self.net(x), p=2, dim=1)


# ── CKA metric (logged, not used in loss) ──────────────────────────────────────
def linear_cka(X: torch.Tensor, Y: torch.Tensor) -> float:
    def centering(K):
        n = K.size(0)
        H = torch.eye(n, device=K.device) - (1.0 / n) * torch.ones(n, n, device=K.device)
        return H @ K @ H

    def hsic(Kc, Lc):
        return float((Kc * Lc).sum() / ((Kc.size(0) - 1) ** 2))

    with torch.no_grad():
        X = X - X.mean(0, keepdim=True)
        Y = Y - Y.mean(0, keepdim=True)
        Kxc = centering(X @ X.t())
        Kyc = centering(Y @ Y.t())
        num   = hsic(Kxc, Kyc)
        denom = np.sqrt(max(hsic(Kxc, Kxc), 1e-10) * max(hsic(Kyc, Kyc), 1e-10))
        return float(num / denom)


# ── Data ────────────────────────────────────────────────────────────────────────
def get_train_loader():
    transform = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    ds = torchvision.datasets.CIFAR10(root="../../data", train=True,
                                       download=True, transform=transform)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True,
                      num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

def get_test_loader():
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    ds = torchvision.datasets.CIFAR10(root="../../data", train=False,
                                       download=True, transform=transform)
    return DataLoader(ds, batch_size=512, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)


# ── Helpers ─────────────────────────────────────────────────────────────────────
def model_size_mb(model: nn.Module) -> float:
    with tempfile.NamedTemporaryFile(suffix=".pth", delete=False) as f:
        tmp = f.name
    try:
        torch.save(model.state_dict(), tmp)
        return os.path.getsize(tmp) / 1024 ** 2
    finally:
        os.remove(tmp)

def param_count_nonzero(model: nn.Module) -> int:
    return int(sum((p != 0).sum().item() for p in model.parameters()))

def sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader) -> dict:
    model.eval()
    preds, labels = [], []
    for inputs, lbls in loader:
        inputs = inputs.to(DEVICE, non_blocking=True)
        out    = model(inputs)
        logits = out[0] if isinstance(out, tuple) else out
        preds.extend(logits.argmax(1).cpu().numpy())
        labels.extend(lbls.numpy())
    yp, yt = np.array(preds), np.array(labels)
    return {
        "accuracy" : float(accuracy_score(yt, yp)),
        "precision": float(precision_score(yt, yp, average="macro", zero_division=0)),
        "recall"   : float(recall_score(yt, yp, average="macro", zero_division=0)),
        "f1"       : float(f1_score(yt, yp, average="macro", zero_division=0)),
    }

def measure_inference_detailed(model: nn.Module, device: torch.device) -> dict:
    model  = copy.deepcopy(model).eval().to(device)
    is_gpu = device.type == "cuda"

    # ── Single image ──────────────────────────────────────────────────────────
    dummy_single = torch.randn(1, 3, 32, 32, device=device)
    with torch.no_grad():
        for _ in range(50): model(dummy_single)
    if is_gpu: torch.cuda.synchronize()

    times = []
    with torch.no_grad():
        for _ in range(500):
            if is_gpu: torch.cuda.synchronize()
            t0 = time.perf_counter()
            model(dummy_single)
            if is_gpu: torch.cuda.synchronize()
            times.append(time.perf_counter() - t0)
    single_ms = float(np.mean(times) * 1000)

    # ── Batch 128 ─────────────────────────────────────────────────────────────
    dummy_batch = torch.randn(128, 3, 32, 32, device=device)
    if is_gpu:
        start_ev = torch.cuda.Event(enable_timing=True)
        end_ev   = torch.cuda.Event(enable_timing=True)
        with torch.no_grad():
            for _ in range(10): model(dummy_batch)
        torch.cuda.synchronize()
        batch_times = []
        with torch.no_grad():
            for _ in range(100):
                start_ev.record()
                model(dummy_batch)
                end_ev.record()
                torch.cuda.synchronize()
                batch_times.append(start_ev.elapsed_time(end_ev))
        batch_ms = float(np.mean(batch_times))
    else:
        with torch.no_grad():
            for _ in range(5): model(dummy_batch)
        t0 = time.perf_counter()
        with torch.no_grad():
            for _ in range(20): model(dummy_batch)
        batch_ms = (time.perf_counter() - t0) / 20 * 1000

    return {
        "single_img_gpu_ms"  : round(single_ms, 4),
        "batch128_gpu_ms"    : round(batch_ms, 4),
        "per_img_gpu_ms"     : round(batch_ms / 128, 4),
        "throughput_imgs_sec": round(128 / (batch_ms / 1000), 1),
        "timing_method"      : "CUDA events + torch.cuda.synchronize()",
    }

def compute_flops(model: nn.Module) -> float:
    from thop import profile as thop_profile
    m = copy.deepcopy(model).cpu().eval()
    dummy = torch.randn(1, 3, 32, 32)
    try:
        macs, _ = thop_profile(m, inputs=(dummy,), verbose=False)
        return round(float(macs * 2 / 1e9), 4)
    except Exception as e:
        print(f"      [flops] WARNING: thop failed ({e})", flush=True)
        return None


# ══════════════════════════════════════════════════════════════════════════════
# Projection distillation training loop
# ══════════════════════════════════════════════════════════════════════════════
def run_projection_kd(teacher_raw: nn.Module,
                      train_loader: DataLoader,
                      test_loader: DataLoader,
                      baseline_acc: float) -> dict:

    print(f"\n  ┌─ [projection / proj_dim={PROJ_DIM} / beta={BETA} / "
          f"tau={TAU} / epochs={KD_EPOCHS}]", flush=True)

    try:
        student_raw = build_mobilenetv2_cifar(NUM_CLASSES).to(DEVICE)

        teacher_extractor = EmbeddingExtractor(teacher_raw).to(DEVICE)
        student_extractor = MobileNetEmbeddingExtractor(student_raw).to(DEVICE)
        proj_t = ProjectionHead(T_DIM, PROJ_DIM).to(DEVICE)
        proj_s = ProjectionHead(S_DIM, PROJ_DIM).to(DEVICE)
        print(f"      [proj] T_DIM={T_DIM}→{PROJ_DIM}  S_DIM={S_DIM}→{PROJ_DIM}",
              flush=True)

        params = (list(student_raw.parameters()) +
                  list(proj_s.parameters()) +
                  list(proj_t.parameters()))
        optimizer = torch.optim.SGD(params, lr=LR, momentum=0.9,
                                    weight_decay=1e-4, nesterov=True)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=KD_EPOCHS)
        scaler    = torch.amp.GradScaler(enabled=USE_AMP)

        if DEVICE.type == "cuda":
            torch.cuda.reset_peak_memory_stats()

        epoch_train_acc  = []
        epoch_align_loss = []
        epoch_cka        = []
        t_start = time.perf_counter()

        teacher_raw.eval()

        for epoch in range(KD_EPOCHS):
            student_raw.train()
            proj_s.train()
            proj_t.train()

            correct = total = 0
            align_sum      = 0.0
            cka_batch_vals = []
            t_ep = time.perf_counter()

            for batch_idx, (inputs, targets) in enumerate(train_loader):
                inputs  = inputs.to(DEVICE, non_blocking=True)
                targets = targets.to(DEVICE, non_blocking=True)

                optimizer.zero_grad(set_to_none=True)

                with torch.amp.autocast(device_type=DEVICE.type, enabled=USE_AMP):
                    with torch.no_grad():
                        t_logits, t_emb = teacher_extractor(inputs)

                    s_logits, s_emb = student_extractor(inputs)
                    ce_loss   = F.cross_entropy(s_logits, targets)
                    z_t       = proj_t(t_emb.detach())
                    z_s       = proj_s(s_emb)
                    proj_loss = F.mse_loss(z_s, z_t.detach())
                    loss      = ce_loss + BETA * proj_loss

                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

                correct   += (s_logits.detach().argmax(1) == targets).sum().item()
                total     += targets.size(0)
                align_sum += proj_loss.item() * targets.size(0)

                # Log CKA every 50 batches (not used in gradient)
                if (batch_idx + 1) % 50 == 0:
                    with torch.no_grad():
                        cka_val = linear_cka(t_emb[:32].float(), s_emb[:32].float())
                        cka_batch_vals.append(cka_val)

                if (batch_idx + 1) % 100 == 0:
                    elapsed = time.perf_counter() - t_ep
                    done    = batch_idx + 1
                    eta     = elapsed / done * (len(train_loader) - done)
                    print(f"      [epoch {epoch+1}/{KD_EPOCHS}] "
                          f"batch {done}/{len(train_loader)}  "
                          f"acc={correct/total:.4f}  "
                          f"proj_loss={align_sum/total:.5f}  ETA={eta:.0f}s", flush=True)

            scheduler.step()
            sync()

            acc = correct / total
            epoch_train_acc.append(round(acc, 6))
            epoch_align_loss.append(round(align_sum / total, 6))
            if cka_batch_vals:
                epoch_cka.append(round(float(np.mean(cka_batch_vals)), 4))

            ep_time   = time.perf_counter() - t_ep
            remaining = KD_EPOCHS - (epoch + 1)
            cka_str   = f"  CKA={epoch_cka[-1]:.3f}" if epoch_cka else ""
            print(f"      [epoch {epoch+1}/{KD_EPOCHS}] DONE  "
                  f"acc={acc:.4f}  proj_loss={align_sum/total:.5f}{cka_str}  "
                  f"time={ep_time:.1f}s  ETA_remaining={ep_time*remaining:.0f}s",
                  flush=True)

        sync()
        train_time_s = time.perf_counter() - t_start
        peak_gpu_mb  = (round(torch.cuda.max_memory_allocated() / 1024**2, 2)
                        if DEVICE.type == "cuda" else None)

        # Clean up hooks
        teacher_extractor.remove_hook()
        student_extractor.remove_hook()

        # ── Evaluation ────────────────────────────────────────────────────────
        print("      [eval] Classification metrics ...", flush=True)
        metrics = evaluate(student_raw, test_loader)

        print("      [eval] Inference timing ...", flush=True)
        inference_info = measure_inference_detailed(student_raw, DEVICE)

        print("      [eval] FLOPs ...", flush=True)
        flops_g = compute_flops(student_raw)

        size_mb   = model_size_mb(student_raw)
        params_nz = param_count_nonzero(student_raw)
        acc_drop  = baseline_acc - metrics["accuracy"]

        print(f"  └─ Acc={metrics['accuracy']:.4f}  Drop={acc_drop:+.4f}  "
              f"Size={size_mb:.2f}MB  "
              f"Throughput={inference_info['throughput_imgs_sec']:.0f} img/s  "
              f"Train={train_time_s:.1f}s", flush=True)

        return {
            "sweep"             : "best_config",
            "method"            : METHOD,
            "temperature"       : TAU,
            "alpha"             : ALPHA,
            "beta"              : BETA,
            "gamma"             : GAMMA,
            "proj_dim"          : PROJ_DIM,
            "proj_temp"         : PROJ_TEMP,
            "lr"                : LR,
            "lr_schedule"       : "cosine",
            "epochs"            : KD_EPOCHS,
            "train_device"      : str(DEVICE),
            "use_amp"           : USE_AMP,
            "train_time_s"      : round(train_time_s, 2),
            "epoch_train_acc"   : epoch_train_acc,
            "epoch_align_loss"  : epoch_align_loss,
            "epoch_cka"         : epoch_cka,
            "accuracy"          : round(metrics["accuracy"],  6),
            "precision"         : round(metrics["precision"], 6),
            "recall"            : round(metrics["recall"],    6),
            "f1"                : round(metrics["f1"],        6),
            "accuracy_drop"     : round(acc_drop, 6),
            "params_nonzero"    : params_nz,
            "size_mb"           : round(size_mb, 4),
            "flops_G"           : flops_g,
            "inference_ms"      : inference_info,
            "peak_gpu_memory_mb": peak_gpu_mb,
            "description"       : (
                f"Cross-Arch KD ResNet50→MobileNetV2 ({METHOD}, "
                f"proj_dim={PROJ_DIM}, beta={BETA}, tau={TAU}, epochs={KD_EPOCHS})"
            ),
            "status": "ok",
        }

    except Exception as e:
        import traceback
        print(f"  └─ FAILED: {e}", flush=True)
        traceback.print_exc()
        return {
            "method": METHOD, "proj_dim": PROJ_DIM,
            "status": "failed", "error": str(e),
            "accuracy": None, "accuracy_drop": None,
        }


# ── Main ────────────────────────────────────────────────────────────────────────
def main():
    print(f"\n{'='*65}")
    print("  Cross-Architecture KD — Projection (Best Config)")
    print(f"  Teacher: ResNet-50  →  Student: MobileNetV2")
    print(f"  method={METHOD} | proj_dim={PROJ_DIM} | beta={BETA} | "
          f"tau={TAU} | epochs={KD_EPOCHS}")
    print(f"  device={DEVICE} | batch={BATCH_SIZE} | AMP={USE_AMP}")
    print(f"{'='*65}\n")

    with open(BASELINE_METRICS_PATH) as f:
        baseline = json.load(f)
    baseline_acc = baseline["accuracy"]
    print(f"  Baseline teacher acc : {baseline_acc:.4f}\n", flush=True)

    teacher     = load_teacher(TEACHER_MODEL_PATH)
    student_ref = build_mobilenetv2_cifar(NUM_CLASSES)

    teacher_size_mb = model_size_mb(teacher)
    student_size_mb = model_size_mb(student_ref)
    teacher_params  = param_count_nonzero(teacher)
    student_params  = param_count_nonzero(student_ref)

    print(f"  Teacher (ResNet-50)   — Size: {teacher_size_mb:.2f} MB  "
          f"Non-zero params: {teacher_params:,}")
    print(f"  Student (MobileNetV2) — Size: {student_size_mb:.2f} MB  "
          f"Non-zero params: {student_params:,}  "
          f"({student_params/teacher_params*100:.1f}% of teacher)\n", flush=True)

    train_loader = get_train_loader()
    test_loader  = get_test_loader()

    result = run_projection_kd(teacher, train_loader, test_loader, baseline_acc)

    report = {
        "method"      : "cross_architecture_kd_projection_best",
        "description" : (
            f"Cross-Arch KD ResNet-50→MobileNetV2: projection, "
            f"proj_dim={PROJ_DIM}, beta={BETA}, tau={TAU}, epochs={KD_EPOCHS}"
        ),
        "teacher_arch": "ResNet-50 (CIFAR-10 modified)",
        "student_arch": "MobileNetV2 (CIFAR-10 modified)",
        "dataset"     : "CIFAR-10",
        "train_device": str(DEVICE),
        "use_amp"     : USE_AMP,
        "kd_epochs"   : KD_EPOCHS,
        "baseline"    : baseline,
        "teacher_info": {
            "size_mb"       : round(teacher_size_mb, 4),
            "params_nonzero": teacher_params,
        },
        "student_info": {
            "size_mb"       : round(student_size_mb, 4),
            "params_nonzero": student_params,
            "compression_ratio": round(teacher_params / student_params, 2),
        },
        "result": result,
    }

    with open(OUTPUT_JSON, "w") as f:
        json.dump(report, f, indent=2)

    print(f"\n  ✓ Saved → {OUTPUT_JSON}")
    if result.get("accuracy") is not None:
        print(f"  Acc={result['accuracy']:.4f}  |  Drop={result['accuracy_drop']:+.4f}  |  "
              f"F1={result['f1']:.4f}  |  "
              f"Size={result['size_mb']:.2f}MB vs teacher {teacher_size_mb:.2f}MB  |  "
              f"Throughput={result['inference_ms']['throughput_imgs_sec']:.0f} img/s  |  "
              f"FLOPs={result['flops_G']} G")


if __name__ == "__main__":
    main()

[init] PyTorch 2.12.0.dev20260324+cu128
[init] Device  : cuda
[init] GPU     : NVIDIA GeForce RTX 5070 Laptop GPU
[init] AMP     : True
[init] Config  : projection | proj_dim=64 | beta=1.0 | tau=4.0 | epochs=10

  Cross-Architecture KD — Projection (Best Config)
  Teacher: ResNet-50  →  Student: MobileNetV2
  method=projection | proj_dim=64 | beta=1.0 | tau=4.0 | epochs=10
  device=cuda | batch=128 | AMP=True

  Baseline teacher acc : 0.9320

[load] Loading teacher from ../../__2__baseline_resnet50_cifar10.pth ...
[load] Teacher OK
  Teacher (ResNet-50)   — Size: 90.03 MB  Non-zero params: 23,520,842
  Student (MobileNetV2) — Size: 8.76 MB  Non-zero params: 2,219,626  (9.4% of teacher)


  ┌─ [projection / proj_dim=64 / beta=1.0 / tau=4.0 / epochs=10]
      [proj] T_DIM=2048→64  S_DIM=1280→64
      [epoch 1/10] batch 100/391  acc=0.2052  proj_loss=0.02608  ETA=28s
      [epoch 1/10] batch 200/391  acc=0.2666  proj_loss=0.02169  ETA=15s
      [epoch 1/10] batch 300/391  acc=0.3065  proj